In [ ]:
!pip install flask flask-cors firebase-admin tensorflow pandas numpy joblib scikit-learn pyngrok

import os
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from flask import Flask, jsonify, request
from flask_cors import CORS
import firebase_admin
from firebase_admin import credentials, db


# 1. CARREGAR ARQUIVOS NECESSARIOS ( serviceAccountKey.json, modelo_lstm_energia.keras,Saving scaler_energia.pkl)

from google.colab import files

uploaded = files.upload()

# 1. TOKEN DE AUTENTICAÇÃO DO NGROK

from pyngrok import ngrok

ngrok.set_auth_token("3E6FIuJGibkJKDB6xQoe9mEXqRW_6sfmiqdyYkc8xfB36qBnk")


# 1. Inicialização do Flask e Configuração de CORS
app = Flask(__name__)
CORS(app)  # Permite que o seu Front-end (index.html) faça requisições sem bloqueios de segurança


# 2. Conexão com o Firebase Realtime Database
# Certifique-se de que o ficheiro 'serviceAccountKey.json' está no mesmo diretório
if not firebase_admin._apps:
    cred = credentials.Certificate("serviceAccountKey.json")# <-- COLOQUE sua credenciais geradas do Firebase
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://consumo-conciente-default-rtdb.firebaseio.com/'  # <-- COLOQUE A URL DO SEU FIREBASE AQUI
    })

# 3. Carregamento dos ficheiros de Inteligência Artificial treinados
MODELO_PATH = 'modelo_lstm_energia.keras'
SCALER_PATH = 'scaler_energia.pkl'

print("A carregar o modelo LSTM e o Scaler...")
modelo = tf.keras.models.load_model(MODELO_PATH)
scaler = joblib.load(SCALER_PATH)
print("Modelos carregados com sucesso!")

# Definindo exatamente as variáveis solicitadas
COLUNAS_MONITORADAS = ['consumo', 'corrente', 'potencia', 'voltagem']
INDICE_ALVO = COLUNAS_MONITORADAS.index('consumo')
JANELA_PADRAO = 60

# 4. Função Auxiliar para Aplanar e Tratar os Dados do Firebase
def buscar_dados_pzem(quantidade):
    """Busca o histórico do Firebase e aplana a estrutura hierárquica (Data -> Hora)"""
    ref = db.reference('leituras') # Aponta para o nó principal 'leituras' do seu JSON
    dados = ref.get()

    if not dados:
        return pd.DataFrame()

    linhas_achatadas = []

    # Ordena as datas para garantir a ordem cronológica correta
    for data in sorted(dados.keys()):
        horas = dados[data]
        if isinstance(horas, dict):
            # Ordena as horas dentro de cada data
            for hora in sorted(horas.keys()):
                atributos = horas[hora]
                if isinstance(atributos, dict):
                    # Cria uma cópia para não alterar o banco original em memória
                    registro = atributos.copy()

                    # Padronização de nomes: se houver 'consumo_acumulado', mapeia para 'consumo'
                    if 'consumo_acumulado' in registro and 'consumo' not in registro:
                        registro['consumo'] = registro['consumo_acumulado']

                    linhas_achatadas.append(registro)

    # Cria o DataFrame a partir da lista plana
    df = pd.DataFrame(linhas_achatadas)

    # Validação: Garante que todas as colunas exigidas existem no DataFrame (evita KeyError)
    for col in COLUNAS_MONITORADAS:
        if col not in df.columns:
            df[col] = 0.0

    # Filtra apenas as colunas que a rede LSTM conhece e retorna as últimas N amostras (Janela)
    df_filtrado = df[COLUNAS_MONITORADAS].dropna()
    return df_filtrado.tail(quantidade)

# 5. Algoritmo de Previsão Autorregressiva Dinâmica para N passos
def prever_passos_futuros(df, passos_solicitados):
    """Aplica o modelo LSTM de forma iterativa para prever N passos à frente"""
    # Normaliza a janela de entrada com o MinMaxScaler carregado
    dados_normalizados = scaler.transform(df[COLUNAS_MONITORADAS])
    janela_atual = dados_normalizados.copy()
    previsoes_finais = []

    for _ in range(passos_solicitados):
        # Altera o formato para o padrão exigido pelo Keras/TensorFlow: (1, 60, 4)
        input_lstm = np.reshape(janela_atual, (1, janela_atual.shape[0], janela_atual.shape[1]))

        # Faz a predição do próximo ponto (retorna o valor escalonado)
        predicao_escalonada = modelo.predict(input_lstm, verbose=0)

        # Cria uma linha temporária para aplicar a inversão do scaler de forma correta
        linha_fake = np.zeros((1, len(COLUNAS_MONITORADAS)))
        linha_fake[0, INDICE_ALVO] = predicao_escalonada[0, 0]

        # Desnormaliza o valor obtido para voltar à escala real (kWh)
        valor_real_consumo = scaler.inverse_transform(linha_fake)[0, INDICE_ALVO]
        previsoes_finais.append(float(valor_real_consumo))

        # --- Atualização da Janela Temporal ---
        # Mantém as variáveis de contexto da última leitura e atualiza apenas o consumo com a previsão
        proxima_linha = janela_atual[-1].copy()
        proxima_linha[INDICE_ALVO] = predicao_escalonada[0, 0]

        # Remove o primeiro registo (mais antigo) e insere o novo registo no fim da matriz
        janela_atual = np.vstack([janela_atual[1:], proxima_linha])

    return previsoes_finais

# --- ENDPOINTS DA API REST ---

@app.route('/api/status_atual', methods=['GET'])
def obter_status_atual():
    """Retorna os dados em tempo real mais recentes do sensor e a previsão imediata (+1 passo)"""
    df = buscar_dados_pzem(JANELA_PADRAO)

    if len(df) < JANELA_PADRAO:
        return jsonify({
            'erro': f'Histórico insuficiente no Firebase. São necessárias {JANELA_PADRAO} amostras, mas o sistema possui apenas {len(df)}.'
        }), 400

    # Obtém o último registo real guardado pelo PZEM
    ultima_leitura = df.iloc[-1].to_dict()
    # Gera a previsão para o próximo passo imediato
    previsao_imediata = prever_passos_futuros(df, passos_solicitados=1)

    return jsonify({
        'status': 'sucesso',
        'leitura_atual': ultima_leitura,
        'previsao_proximo_passo': previsao_imediata[0]
    })

@app.route('/api/previsao_futura', methods=['GET'])
def obter_previsao_futura():
    """Retorna a projeção de N passos à frente definida dinamicamente pelo parâmetro '?passos=' na URL"""
    passos = int(request.args.get('passos', 5)) # Se não for enviado parâmetros, assume 5 passos como padrão
    df = buscar_dados_pzem(JANELA_PADRAO)

    if len(df) < JANELA_PADRAO:
        return jsonify({'erro': 'Histórico insuficiente para realizar previsões futuras.'}), 400

    projecao = prever_passos_futuros(df, passos_solicitados=passos)

    return jsonify({
        'status': 'sucesso',
        'passos_calculados': passos,
        'valores_projetados': projecao
    })



 # @app.route('/', methods=['GET'])
 #def testar_servidor():
  #return jsonify({'mensagem': 'API de Eficiência Energética rodando com sucesso!'})

#if __name__ == '__main__':


public_url = ngrok.connect(5000)

print("URL pública da API:")
print(public_url)


 # Se rodar localmente: porta 5000. Se rodar no Colab, o ngrok fará o encaminhamento a partir daqui.

app.run(
    host="0.0.0.0",
    port=5000,
    debug=False,
    use_reloader=False
)



Saving modelo_lstm_energia.keras to modelo_lstm_energia (2).keras
Saving scaler_energia.pkl to scaler_energia (2).pkl
Saving serviceAccountKey.json to serviceAccountKey (2).json
A carregar o modelo LSTM e o Scaler...
Modelos carregados com sucesso!
URL pública da API:
NgrokTunnel: "https://giggling-dormitory-tusk.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 18:44:16] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 18:52:27] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 18:52:30] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 18:53:05] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 18:53:46] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 19:12:59] "GET /api/status_atual HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 19:14:46] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 19:15:29] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 19:15:31] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/20